# 07 PR 曲线、AP 与 mAP

这一课承接上一课的 TP、FP、FN、Precision 和 Recall。我们要解决的新问题是：**置信度阈值有很多种选法，不能只挑一个阈值评价模型，那么怎样把模型在各种阈值下的表现汇总起来？**

整条来龙去脉是：

$$
\text{预测结果按分数排序}
\longrightarrow \text{逐步放入预测}
\longrightarrow \text{得到一系列 Precision、Recall}
\longrightarrow \text{画出 PR 曲线}
\longrightarrow \text{计算 AP}
\longrightarrow \text{对类别和 IoU 阈值求平均得到 mAP}
$$

## 一、先把上一课放回主线中

上一课中，我们先固定两个条件：

1. 一个**分数阈值**，决定哪些预测框被保留；
2. 一个**评价 IoU 阈值**，帮助判断保留下来的预测能否匹配真实目标。

匹配结束后，才统计 TP、FP、FN，再计算：

$$
\operatorname{Precision}=\frac{TP}{TP+FP}
$$

$$
\operatorname{Recall}=\frac{TP}{TP+FN}
$$

所以你之前说它像混淆矩阵那一套，是有道理的：**统计名称相同，都是在数预测对了多少、错了多少、漏了多少。**

但目标检测多了一道关键步骤：一张图里有多个预测框和多个真实框，必须先根据类别、IoU 和一对一匹配关系，确定每个预测框究竟对应谁，才能统计 TP、FP、FN。

## 二、为什么只看一个分数阈值不够

假设模型给出了许多预测框，每个预测框都有一个置信度分数。部署模型时，我们可能规定：

$$
s \ge 0.7
$$

才保留预测。可是把阈值改成 $0.5$，保留下来的框就会变多，Precision 和 Recall 也会跟着改变。

一般来说：

- 分数阈值高：留下的预测少而谨慎，通常 Precision 较高，但容易漏检，Recall 较低；
- 分数阈值低：留下的预测多而大胆，通常 Recall 较高，但也容易带入 FP，Precision 可能下降。

因此，只报告某一个人为选定的分数阈值，会掩盖模型在其他阈值下的表现。PR 曲线就是用来观察**阈值不断变化时，Precision 和 Recall 怎样一起变化**。

## 三、继续使用上一课的检测结果

为了只关注主线，我们先评价一个类别：狗。整个验证集中共有 3 只真实的狗。模型给出了 5 条狗类别预测，按置信度从高到低排列：

| 排名 | 预测 | 置信度 | 与真实目标匹配后的结果 | 原因 |
|---:|---|---:|---|---|
| 1 | $P_1$ | $0.95$ | TP | 正确检测到第一只狗 |
| 2 | $P_2$ | $0.90$ | TP | 正确检测到第二只狗 |
| 3 | $P_3$ | $0.70$ | FP | 重复检测了已经被匹配的狗 |
| 4 | $P_4$ | $0.60$ | FP | 定位太差，未达到评价 IoU 阈值 |
| 5 | $P_5$ | $0.45$ | TP | 正确检测到第三只狗 |

这里的 TP 和 FP 不是看分数直接决定的。分数只决定排列顺序；TP 或 FP 是在固定评价规则后，通过预测框与真实框匹配得到的。

## 四、降低分数阈值，就是逐条放入预测

先把分数阈值设得比 $0.95$ 还高，一个预测都不保留。然后逐渐降低阈值：

$$
\text{没有预测}
\longrightarrow P_1
\longrightarrow P_1,P_2
\longrightarrow P_1,P_2,P_3
\longrightarrow \cdots
$$

因为预测已经按分数从高到低排好，所以每多考虑一条预测，就等价于把分数阈值降低到能够接纳这条预测的位置。

每放入一条预测，我们都重新统计截至当前位置的累计 TP 和累计 FP。

## 五、逐行计算累计 Precision 和 Recall

真实目标总数固定为 3，因此在任意一步都有：

$$
FN=3-TP
$$

逐条加入预测后得到：

| 加入到排名 | 当前结果 | 累计 TP | 累计 FP | Precision | Recall |
|---:|---|---:|---:|---:|---:|
| 1 | TP | 1 | 0 | $1/1=1.000$ | $1/3=0.333$ |
| 2 | TP | 2 | 0 | $2/2=1.000$ | $2/3=0.667$ |
| 3 | FP | 2 | 1 | $2/3=0.667$ | $2/3=0.667$ |
| 4 | FP | 2 | 2 | $2/4=0.500$ | $2/3=0.667$ |
| 5 | TP | 3 | 2 | $3/5=0.600$ | $3/3=1.000$ |

这张表就是 PR 曲线的原始数据。每一行对应一个分数阈值位置。

### 先读懂表中的三个关键变化

加入 $P_1$ 和 $P_2$ 时，新增的都是 TP，所以 Recall 从 $0$ 增长到 $2/3$，Precision 仍然保持为 $1$。

加入 $P_3$ 和 $P_4$ 时，新增的都是 FP。此时没有找到新的真实目标，所以 Recall 不变；错误预测增多，所以 Precision 下降。

加入 $P_5$ 时，终于找到了第三只狗，因此 Recall 上升到 $1$。虽然 Precision 从 $0.5$ 回升到 $0.6$，但此前的两个 FP 并不会消失。

这也说明：**随着阈值降低，Recall 只会不变或上升；原始 Precision 则可能下降，也可能暂时回升。**

## 六、PR 曲线到底是什么

PR 是 Precision–Recall 的缩写。画图时通常规定：

$$
x=\operatorname{Recall},\qquad y=\operatorname{Precision}
$$

于是上表中的每一行都会形成一个点：

$$
(0.333,1.000),\ (0.667,1.000),\ (0.667,0.667),\ (0.667,0.500),\ (1.000,0.600)
$$

把这些点按照阈值降低的顺序连接起来，就得到 PR 曲线。它描述的不是模型在某一个固定分数阈值下的表现，而是模型从谨慎到大胆的完整取舍过程。

## 七、怎样直观判断一条 PR 曲线好不好

理想模型希望同时做到：

- Recall 高：尽量找全真实目标；
- Precision 高：保留下来的预测尽量正确。

因此，PR 曲线越靠近右上方通常越好。直观地说，好模型能在 Recall 不断提高的同时，让 Precision 尽量晚一些下降。

但是一整条曲线不方便直接比较多个模型，于是需要把曲线压缩成一个数，这个数就是 AP。

## 八、先用一句人话理解 AP

**AP 代表：模型在努力把物体找全的过程中，能不能一直少报错。**

可以把所有预测框看成一支队伍。模型最有把握的框排在最前面，没那么有把握的框排在后面。我们从队伍前面开始，一个一个查看：

- 如果前面大多是正确框，说明模型不仅会找，而且知道哪些预测最靠谱，AP 就高；
- 如果错误框很早就混进来，或者为了找全目标必须接纳大量错误框，AP 就低。

所以，高 AP 可以理解成：**找得比较全、错得比较少、正确框通常还排得靠前。**

低 AP 则表示至少存在一种明显问题：漏掉很多目标、错误框很多、定位不够准，或者模型把错误框打了过高的分数。

### 结合图片理解：为什么面积大就是 AP 高

![高 AP 与低 AP 的 PR 曲线对比](images/07_ap_high_low.png)

看这张图时，按下面三步来，不需要先计算面积：

1. 先看横轴 Recall：越往右，表示找到的真实目标越多；
2. 再看纵轴 Precision：越靠上，表示当前保留的预测越可靠；
3. 最后看曲线下面的涂色面积：面积越大，AP 越高。

左图中，曲线向右移动时一直保持在较高的位置。这表示模型在逐渐找全目标的同时，没有迅速增加大量错误预测，所以涂色面积大，AP 高。

右图中，曲线很早就下降了。这表示模型为了多找一些目标，很快就带进来许多错误预测，所以涂色面积小，AP 低。

图下方的两排圆点都包含 3 个 TP 和 2 个 FP，但左边把 TP 排在前面，右边却让 FP 很早出现。因此 AP 不只看最终对了几个，还看模型是否把真正可靠的预测排在前面。

AP 是 Average Precision。它用 PR 曲线下的面积，把整条曲线压缩成一个分数：

$$
AP=\int_0^1 p(r)\,dr
$$

其中 $r$ 表示 Recall，$p(r)$ 表示该 Recall 位置对应的 Precision。

面积越大，说明模型在从低 Recall 走向高 Recall 的过程中，能够更长时间维持较高 Precision。

最重要的限定是：**一个 AP 对应一个类别，并且对应一个固定的评价 IoU 阈值。**例如，狗类别在评价 IoU 阈值 $0.5$ 下会得到一个 AP。

### Average Precision 不是把表里的 Precision 做算术平均

名字里的 Average 很容易让人误以为：

$$
AP \overset{?}{=} \frac{1.000+1.000+0.667+0.500+0.600}{5}
$$

这不是 AP 的定义。

AP 关注的是 **Precision 随 Recall 展开的曲线面积**。如果加入一个 FP 后 Recall 没变，虽然表中多了一行 Precision，但它没有带来新的 Recall 区间，不能简单地与其他行等权平均。

## 九、为什么常对 Precision 做插值

原始 Precision 可能忽高忽低。评价协议通常会先构造一个更稳定的插值 Precision：在某个 Recall 位置，查看这个位置及其右侧能够达到的最高 Precision。

$$
p_{\mathrm{interp}}(r)=\max_{\tilde r\ge r}p(\tilde r)
$$

它的意思不是修改模型预测，而是在计算面积时，用右侧最佳 Precision 形成一个不向上反弹的包络线。

这样做以后，Recall 越高，插值 Precision 只会保持不变或下降，更适合稳定地计算曲线面积。

## 十、用当前例子计算一个简化 AP

在我们的例子中：

- Recall 从 $0$ 增加到 $1/3$ 时，可达到的插值 Precision 为 $1$；
- Recall 从 $1/3$ 增加到 $2/3$ 时，可达到的插值 Precision 仍为 $1$；
- Recall 从 $2/3$ 增加到 $1$ 时，对应的插值 Precision 为 $0.6$。

因此，用分段矩形面积演示：

$$
\begin{aligned}
AP
&=\frac{1}{3}\times1
+\frac{1}{3}\times1
+\frac{1}{3}\times0.6\\
&=0.8667
\end{aligned}
$$

所以，这个类别在当前评价 IoU 阈值下的 AP 约为 $0.867$。

> 这里使用的是便于理解的插值面积算法。不同数据集和评测工具在采样点、插值和边界处理上可能略有差异，正式实验必须遵循对应评测协议。

## 十一、AP 真正在奖励什么

AP 不只关心最终找到了几个目标，还关心正确预测在置信度排序中出现得够不够早。

假设同样是 3 个 TP 和 2 个 FP：

- 排序 A：TP、TP、TP、FP、FP；
- 排序 B：FP、TP、FP、TP、TP。

两种排序最终的 TP 和 FP 数量相同，但是排序 A 会得到更大的 AP，因为模型把真正可靠的预测排在了前面。

所以置信度分数的作用不仅是能否超过某个阈值，还包括：**能否把正确预测排在错误预测前面。**

## 十二、AP50 与 AP75 中的数字是什么

先固定类别，再固定评价 IoU 阈值，才能计算一条 PR 曲线和一个 AP。

- $AP_{50}$：用 $0.50$ 作为评价 IoU 阈值；
- $AP_{75}$：用 $0.75$ 作为评价 IoU 阈值。

$AP_{75}$ 对定位精度要求更高。一个框在 IoU 为 $0.62$ 时，在 $AP_{50}$ 的评价中可能是 TP，但在 $AP_{75}$ 的评价中会是 FP。

必须特别注意：**$AP_{50}$ 中的 $0.50$ 是评价 IoU 阈值，不是置信度分数阈值。**计算 PR 曲线时，置信度阈值仍然需要从高到低扫描。

## 十三、从 AP 到 mAP

AP 只评价一个类别。如果数据集有 $C$ 个类别，就先分别计算每个类别的 AP，再对类别求平均。

在固定评价 IoU 阈值 $0.5$ 时：

$$
mAP@0.5=\frac{1}{C}\sum_{c=1}^{C}AP_{c,0.5}
$$

例如只有猫、狗两个类别：

$$
AP_{\mathrm{cat},0.5}=0.80,\qquad AP_{\mathrm{dog},0.5}=0.60
$$

那么：

$$
mAP@0.5=\frac{0.80+0.60}{2}=0.70
$$

## 十四、COCO 风格的 mAP 为什么更严格

常见的 COCO 风格主指标会使用一组评价 IoU 阈值：

$$
\mathcal{T}=\{0.50,0.55,0.60,\ldots,0.95\}
$$

每个类别在每个评价 IoU 阈值下都有一个 AP，最后同时对类别和 IoU 阈值求平均：

$$
mAP@0.5{:}0.95
=\frac{1}{C\lvert\mathcal{T}\rvert}
\sum_{c=1}^{C}\sum_{t\in\mathcal{T}}AP_{c,t}
$$

它不只要求模型大致找到目标，还会通过较高的 IoU 阈值检查框得是否准确。因此，同一个模型的 $mAP@0.5{:}0.95$ 通常低于 $mAP@0.5$。

## 十五、三个阈值一定不要混在一起

目标检测中经常同时出现三个阈值：

| 阈值 | 用在什么阶段 | 决定什么 |
|---|---|---|
| 置信度分数阈值 | 输出筛选或构造 PR 曲线 | 哪些预测被保留 |
| 评价 IoU 阈值 | TP、FP 匹配与评价 | 预测框是否足够接近真实框 |
| NMS IoU 阈值 | 后处理 | 两个预测框重叠多大时抑制低分框 |

它们都可能写成 $0.5$，但含义完全不同。

计算一条 PR 曲线时，评价 IoU 阈值通常先固定，而置信度分数阈值不断变化。NMS 阈值属于产生最终预测结果的后处理设置。

## 十六、评价是在整个数据集上累计，不是逐图算完再平均

对某一个类别计算 AP 时，通常会：

1. 收集整个验证集中这个类别的所有预测；
2. 按置信度从高到低统一排序；
3. 在各张图片内部，把预测与各自图片中的真实目标进行匹配；
4. 根据全数据集排序后的 TP、FP 序列计算 PR 曲线和 AP。

预测框不能跨图片匹配真实目标，但统计结果会跨图片累计。

因此，mAP 也不是先为每张图片计算一个 AP，再对图片做平均。它主要是在**类别维度**以及某些协议中的**评价 IoU 阈值维度**上做平均。

## 十七、NMS、重复预测与 AP 的关系

上一课已经看到：同一个真实目标只能匹配一次。对同一只狗给出多个重复框时，通常只有最高分且成功匹配的那个框是 TP，其余重复框会成为 FP。

重复 FP 如果排在前面，会让 Precision 更早下降，从而压低 AP。NMS 的作用之一，就是在进入评价之前减少这类重复预测。

于是传统检测流程中的关系可以串起来：

$$
\text{密集产生许多候选预测}
\longrightarrow \text{分数筛选与 NMS}
\longrightarrow \text{按评价规则匹配 TP、FP}
\longrightarrow \text{PR 曲线与 AP}
$$

## 十八、常见误区

### 误区 1：AP 就是 Accuracy

不是。目标检测通常需要同时评价类别、定位、重复预测和漏检，AP 汇总的是 PR 曲线。

### 误区 2：AP 是所有 Precision 的普通平均

不是。AP 是 Precision 随 Recall 展开的曲线面积。

### 误区 3：$AP_{50}$ 表示只保留置信度大于 $0.5$ 的框

不是。这里的 $0.5$ 是评价 IoU 阈值。

### 误区 4：mAP 是对每张图片的结果求平均

不是。通常先在整个数据集上为每个类别形成 PR 曲线和 AP，再对类别求平均；COCO 风格还会对多个评价 IoU 阈值求平均。

### 误区 5：最终 TP 数量相同，AP 就相同

不一定。置信度排序不同，会让错误预测在 PR 曲线中出现得更早或更晚，因此 AP 可能不同。

## 十九、用一句话区分这些概念

- Precision：当前留下的预测里，有多少是正确的；
- Recall：所有真实目标里，有多少已经被找到；
- PR 曲线：分数阈值变化时，Precision 与 Recall 的完整关系；
- AP：一个类别、一个评价 IoU 阈值下的 PR 曲线面积；
- mAP：对多个类别的 AP 求平均，有些协议还会再对多个评价 IoU 阈值求平均。

## 二十、自我检查

先不要急着看答案，尝试自己回答：

1. 为什么不能只用一个固定置信度阈值下的 Precision 评价模型？
2. 构造 PR 曲线时，哪个阈值不断变化，哪个阈值先固定？
3. 加入一个 FP 时，Recall 为什么不变？
4. AP 为什么不是表中所有 Precision 的算术平均？
5. $AP_{50}$ 中的 $50$ 表示什么？
6. mAP 中的平均通常发生在哪些维度？
7. 两个模型最终 TP、FP 数量相同，为什么 AP 仍可能不同？

### 参考答案

1. 不同分数阈值会产生不同的 Precision 和 Recall，单个阈值只能看到一个工作点。
2. 置信度分数阈值不断变化；类别和评价 IoU 阈值先固定。
3. FP 没有找到新的真实目标，所以累计 TP 不变，Recall 也不变。
4. AP 是 PR 曲线关于 Recall 的面积，不同表格行并不代表等宽的 Recall 区间。
5. 表示评价 IoU 阈值为 $0.50$。
6. 基本 mAP 对类别求平均；COCO 风格主指标还对多个评价 IoU 阈值求平均。
7. AP 还反映置信度排序质量。FP 越早出现，Precision 越早下降，AP 往往越低。

## 二十一、本课主线总结

这一课最核心的逻辑不是背 AP 公式，而是理解评价过程：

$$
\text{先固定类别与评价 IoU 阈值}
\longrightarrow \text{预测按置信度排序}
\longrightarrow \text{逐步降低分数阈值}
\longrightarrow \text{累计 TP、FP}
\longrightarrow \text{得到 PR 曲线}
\longrightarrow \text{曲线面积成为 AP}
\longrightarrow \text{多个 AP 平均成为 mAP}
$$

现在，目标检测的评价主线已经完整：预测框先经过匹配变成 TP、FP，未匹配真实目标形成 FN；它们产生 Precision 和 Recall；扫描分数阈值得到 PR 曲线；曲线面积得到 AP；最后再得到 mAP。

## 下一课：用简化 YOLO 串起完整检测流程

到这里，我们已经学完目标检测的共同零件：边界框、IoU、分类与回归、目标分配、重复预测、NMS 和 mAP。

下一课不追逐 YOLO 的版本细节，而是用一个简化 YOLO 回答：

- 一张图片怎样经过骨干网络变成特征图；
- 特征图上的预测单位怎样输出类别、置信度和边界框；
- 训练时怎样分配正负样本并计算损失；
- 推理时为什么需要分数筛选和 NMS；
- 这些步骤最后怎样接受 mAP 评价。

这样我们就能把此前分散学习的概念放进一次完整的目标检测旅程中，然后再回到 DETR，看它究竟改变了哪些环节。